# Evaluation — Baseline vs Fine-tuned on Test Set
Evaluates both models on the held-out test split and saves all metrics, probabilities, and per-class numbers for Week 5 visualisation.

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd gdrive/My\ Drive/Colab\ Notebooks/emotion_project

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
import json
from datasets import load_from_disk
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, default_data_collator
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
THRESHOLD  = 0.5
NUM_LABELS = 28
LABEL_NAMES = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
print('device:', DEVICE)

## 1. Load test split

In [ ]:
tokenized = load_from_disk('processed/go_emotions_tokenized')
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids'])

class EmotionDataset(Dataset):
    def __init__(self, hf_split, num_labels=NUM_LABELS):
        self.data       = hf_split
        self.num_labels = num_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        label_vec = torch.zeros(self.num_labels, dtype=torch.float)
        for lbl in row['labels']:
            label_vec[lbl] = 1.0
        return {
            'input_ids':      row['input_ids'],
            'attention_mask': row['attention_mask'],
            'token_type_ids': row['token_type_ids'],
            'labels':         label_vec,
        }

test_loader = DataLoader(
    EmotionDataset(tokenized['test']),
    batch_size=64, shuffle=False,
    collate_fn=default_data_collator,
)
print(f'Test examples: {len(tokenized["test"])}')

## 2. Inference helper

In [ ]:
def get_predictions(model, loader):
    """Returns (probs [N,28], labels [N,28]) on the given loader."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids      = batch['input_ids'].to(DEVICE),
                attention_mask = batch['attention_mask'].to(DEVICE),
                token_type_ids = batch['token_type_ids'].to(DEVICE),
            ).logits
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(batch['labels'].numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

## 3. Load baseline model and run inference

In [ ]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    'bhadresh-savani/bert-base-go-emotion'
).to(DEVICE)

baseline_probs, test_labels = get_predictions(baseline_model, test_loader)
print(f'baseline_probs : {baseline_probs.shape}')
print(f'test_labels    : {test_labels.shape}')

## 4. Load fine-tuned model and run inference
Reads `checkpoints/trainer_state.json` to find the best checkpoint automatically.
If you want to evaluate a specific run, set `FINETUNED_CKPT` manually.

In [ ]:
with open('checkpoints/trainer_state.json') as f:
    trainer_state = json.load(f)

FINETUNED_CKPT = trainer_state['best_model_checkpoint']
print(f'Best checkpoint: {FINETUNED_CKPT}')

finetuned_model = AutoModelForSequenceClassification.from_pretrained(FINETUNED_CKPT).to(DEVICE)
finetuned_probs, _ = get_predictions(finetuned_model, test_loader)
print(f'finetuned_probs: {finetuned_probs.shape}')

## 5. Compute all metrics

In [ ]:
def compute_all_metrics(probs, labels, threshold=THRESHOLD):
    preds      = (probs >= threshold).astype(int)
    labels_int = labels.astype(int)

    overall = {
        'micro_f1':    float(f1_score(labels_int, preds, average='micro',    zero_division=0)),
        'macro_f1':    float(f1_score(labels_int, preds, average='macro',    zero_division=0)),
        'weighted_f1': float(f1_score(labels_int, preds, average='weighted', zero_division=0)),
        'exact_match': float((preds == labels_int).all(axis=1).mean()),
    }

    per_class = pd.DataFrame({
        'label':         LABEL_NAMES,
        'precision':     precision_score(labels_int, preds, average=None, zero_division=0),
        'recall':        recall_score(labels_int,    preds, average=None, zero_division=0),
        'f1':            f1_score(labels_int,        preds, average=None, zero_division=0),
        'support':       labels_int.sum(axis=0).astype(int),
        'auc_roc':  [
            float(roc_auc_score(labels_int[:, i], probs[:, i]))
            if labels_int[:, i].sum() > 0 else float('nan')
            for i in range(NUM_LABELS)
        ],
        'avg_precision': [
            float(average_precision_score(labels_int[:, i], probs[:, i]))
            if labels_int[:, i].sum() > 0 else float('nan')
            for i in range(NUM_LABELS)
        ],
    })

    return overall, per_class

In [ ]:
baseline_overall,  baseline_pc  = compute_all_metrics(baseline_probs,  test_labels)
finetuned_overall, finetuned_pc = compute_all_metrics(finetuned_probs, test_labels)

## 6. Comparison table

In [ ]:
metrics = ['micro_f1', 'macro_f1', 'weighted_f1', 'exact_match']
summary = pd.DataFrame({
    'metric':    metrics,
    'baseline':  [baseline_overall[m]  for m in metrics],
    'finetuned': [finetuned_overall[m] for m in metrics],
})
summary['delta'] = summary['finetuned'] - summary['baseline']
summary['delta_pct'] = (summary['delta'] / summary['baseline'] * 100).round(2)

pd.set_option('display.float_format', '{:.4f}'.format)
print('=== Test Set — Overall Metrics ===')
display(summary)

print('\n=== Per-Class Metrics (Fine-tuned, sorted by F1) ===')
display(finetuned_pc.sort_values('f1', ascending=False).reset_index(drop=True))

## 7. Save all results

In [ ]:
os.makedirs('results', exist_ok=True)

# Overall metrics
with open('results/test_metrics_baseline.json',  'w') as f: json.dump(baseline_overall,  f, indent=2)
with open('results/test_metrics_finetuned.json', 'w') as f: json.dump(finetuned_overall, f, indent=2)

# Per-class metrics
baseline_pc.to_csv( 'results/test_per_class_baseline.csv',  index=False)
finetuned_pc.to_csv('results/test_per_class_finetuned.csv', index=False)

# Raw probabilities and labels (needed for ROC / PR curves)
np.save('results/test_probs_baseline.npy',  baseline_probs)
np.save('results/test_probs_finetuned.npy', finetuned_probs)
np.save('results/test_labels.npy',          test_labels)

print('Saved to results/:')
for fname in sorted(os.listdir('results')):
    print(f'  {fname}')